# M9 lab: learning from few runs with Gaussian processes

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M9 · about 45 minutes, plus 60 minutes of practice at the end.

You will rebuild the lesson's Gaussian process in NumPy and check every number against the page with `assert` cells, then compare a standard library:

1. The toy turbine and its records (read from the lesson's data file).
2. A Gaussian process from scratch: kernel, Cholesky factor, mean, band.
3. Settings by maximum likelihood on the page's grid.
4. Scores on 400 held-out records: RMSE, coverage of the bands, pinball loss.
5. Two sources: the maker's curve plus a Gaussian process for the gap; and a case where the cheap curve misleads.
6. scikit-learn's `GaussianProcessRegressor` on the same records.

**How to run it.** Put this notebook and `m09-surrogates-uq.json` (the *Lesson data* button on the lesson page) in one folder. In Google Colab: *File → Upload notebook*, then upload the data file with the folder icon on the left, then *Runtime → Run all*. Locally: Python 3.9 or later with `numpy`, `scipy` and `scikit-learn`; `matplotlib` draws the charts if you have it. Everything runs in under a minute.

**The data.** One invented 2,000 kW turbine, not a real one: records drawn from a stated recipe with fixed seeds. The numbers say nothing about any real turbine.

In [ ]:
import json
from pathlib import Path

import numpy as np
from scipy import stats
from scipy.linalg import cho_solve, solve_triangular

assert Path("m09-surrogates-uq.json").exists(), "m09-surrogates-uq.json not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D = json.load(open("m09-surrogates-uq.json"))
REF, T, G = D["reference"], D["turbine"], D["grid"]
try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except ImportError:
    HAVE_PLT = False

def arr(key):
    a = np.array(D["records"][key], float)
    return a[:, 0], a[:, 1]

v_fit, p_fit = arr("campaign")          # 64 records to fit, used in order
v_test, p_test = arr("test")            # 400 held-out records
print(len(v_fit), "records to fit,", len(v_test), "to score; turbine rated", T["rated"], "kW")

## 1. The toy turbine

- **The maker's curve** $f(v)$: 0 below 3 m/s, $2000\,(v^3 - 3^3)/(11.5^3 - 3^3)$ kW up to 11.5 m/s, 2,000 kW above.
- **The field curve** (hidden from every fit): 0.97 times $f$ averaged over gusts, $E[f(v(1 + 0.12 Z))]$ with $Z$ standard normal.
- **Records**: wind speeds Weibull-distributed between 3 and 20 m/s; power = field curve + scatter of $\sqrt{15^2 + (0.3\,g'(v))^2}$ kW, floored at 0.

The cell below ports the page's gust rule (201 points on $z \in [-5, 5]$) and checks it against the reference block.

In [ ]:
def maker(v):
    v = np.asarray(v, float)
    c, r, P = T["cutIn"], T["ratedSpeed"], T["rated"]
    return np.where(v < c, 0.0, np.where(v >= r, P, P * (v ** 3 - c ** 3) / (r ** 3 - c ** 3)))

GZ = -5 + np.arange(201) / 20
GW = np.exp(-GZ ** 2 / 2); GW /= GW.sum()

def field(v, mode=None):
    v = np.atleast_1d(np.asarray(v, float))
    y = (1 - T["loss"]) * (GW * maker(v[:, None] * (1 + T["ti"] * GZ))).sum(axis=1)
    if mode == "reduce":
        R = T["reduce"]
        y = y * (1 - R["share"] * np.clip((v - R["from"]) / (R["to"] - R["from"]), 0, 1))
    return y

for v, f0, f1 in REF["field"]:
    assert abs(field(v)[0] - f0) < 1e-6 and abs(field(v, "reduce")[0] - f1) < 1e-6
grid = np.linspace(3, 20, 171)
gap = maker(grid) - field(grid)
print(f"largest gap between the maker's and the field curve: {gap.max():.0f} kW at {grid[gap.argmax()]:.1f} m/s")
if HAVE_PLT:
    plt.figure(figsize=(7, 4))
    plt.scatter(v_test, p_test, s=6, c="0.6", label="400 held-out records")
    plt.plot(grid, maker(grid), "--", label="maker's curve"); plt.plot(grid, field(grid), ":", c="k", label="field curve (hidden)")
    plt.xlabel("wind speed, m/s"); plt.ylabel("power, kW"); plt.legend(); plt.show()

## 2. A Gaussian process from scratch

With records $(v_i, y_i)$, a squared-exponential kernel $k(v, v') = s_f^2 \exp(-(v - v')^2 / 2\ell^2)$ and noise $s_n$:

- $K = [k(v_i, v_j)] + s_n^2 I = L L^\top$ (Cholesky);
- the mean's basis $h(v)$ is $[1]$ (one source) or $[1, f(v)/1000]$ (two sources), with weights $\hat\beta = (H^\top K^{-1} H)^{-1} H^\top K^{-1} y$;
- mean $m(v) = h^\top \hat\beta + k_v^\top K^{-1} (y - H\hat\beta)$; the curve's variance $s_f^2 - k_v^\top K^{-1} k_v + u^\top (H^\top K^{-1} H)^{-1} u$ with $u = h - H^\top K^{-1} k_v$; a new record adds $s_n^2$.

The functions below follow the page's code line by line.

In [ ]:
def basis(v, two):
    v = np.atleast_1d(v)
    return np.column_stack([np.ones_like(v), maker(v) / 1000]) if two else np.ones((len(v), 1))

def kern(a, b, hp):
    return hp["sf"] ** 2 * np.exp(-(a[:, None] - b[None, :]) ** 2 / (2 * hp["ell"] ** 2))

def fit(v, y, hp, two):
    n = len(v)
    K = kern(v, v, hp) + hp["sn"] ** 2 * np.eye(n)
    L = np.linalg.cholesky(K)
    H = basis(v, two)
    KiH = cho_solve((L, True), H)
    A = H.T @ KiH
    beta = np.linalg.solve(A, KiH.T @ y)
    r = y - H @ beta
    ri = cho_solve((L, True), r)
    ll = -0.5 * r @ ri - np.log(np.diag(L)).sum() - 0.5 * n * np.log(2 * np.pi)
    return dict(v=v, L=L, H=H, A=A, beta=beta, ri=ri, hp=hp, two=two, ll=ll)

def predict(F, x):
    x = np.atleast_1d(np.asarray(x, float))
    k = kern(x, F["v"], F["hp"])
    h = basis(x, F["two"])
    m = h @ F["beta"] + k @ F["ri"]
    w = solve_triangular(F["L"], k.T, lower=True)
    Kik = cho_solve((F["L"], True), k.T)
    u = h - (F["H"].T @ Kik).T
    lat = F["hp"]["sf"] ** 2 - (w ** 2).sum(axis=0) + np.einsum("ij,ij->i", u @ np.linalg.inv(F["A"]), u)
    lat = np.maximum(lat, 0)
    return m, np.sqrt(lat + F["hp"]["sn"] ** 2)

N0 = D["n0"]
one = fit(v_fit[:N0], p_fit[:N0], D["fits"]["base"]["one"][str(N0)], False)
two = fit(v_fit[:N0], p_fit[:N0], D["fits"]["base"]["two"][str(N0)], True)
assert np.allclose(one["beta"], REF["beta16"]["one"], rtol=1e-7) and np.allclose(two["beta"], REF["beta16"]["two"], rtol=1e-7)
for v, m1, s1, m2, s2 in REF["predict16"]:
    a, b = predict(one, v), predict(two, v)
    assert abs(a[0][0] - m1) < 1e-5 and abs(a[1][0] - s1) < 1e-5 and abs(b[0][0] - m2) < 1e-5 and abs(b[1][0] - s2) < 1e-5
print(f"one source at 16 m/s: {predict(one, 16)[0][0]:.0f} kW ± {1.645 * predict(one, 16)[1][0]:.0f} (90%); the page agrees")
print(f"two sources: rho = {two['beta'][1] / 1000:.3f} times the maker's curve, plus {two['beta'][0]:.0f} kW")

## 3. The records choose: maximum likelihood on the page's grid

The page searches every combination of the grid's length-scales, signal sizes and noise levels (3,808 settings) and keeps the first best in grid order. The cell repeats that for 16 records and for every stored fit (one and two sources, 16 to 64 records, both cases), and checks that each winner beats the runner-up clearly.

In [ ]:
import itertools, time

def search(v, y, two):
    best, second = None, -np.inf
    for ell, sf, sn in itertools.product(G["ell"], G["sf"], G["sn"]):
        try:
            F = fit(v, y, dict(ell=ell, sf=sf, sn=sn), two)
        except np.linalg.LinAlgError:
            continue
        if best is None or F["ll"] > best["ll"]:
            if best is not None:
                second = best["ll"]
            best = F
        elif F["ll"] > second:
            second = F["ll"]
    best["margin"] = best["ll"] - second
    return best

t0 = time.time()
b16 = search(v_fit[:N0], p_fit[:N0], False)
ref = REF["ml16"]
assert (b16["hp"]["ell"], b16["hp"]["sf"], b16["hp"]["sn"]) == (ref["ell"], ref["sf"], ref["sn"]) and abs(b16["ll"] - ref["loglik"]) < 1e-6
print(f"16 records choose ell = {ref['ell']} m/s, sf = {ref['sf']} kW, sn = {ref['sn']} kW (log-likelihood {b16['ll']:.2f}, ahead by {b16['margin']:.3f})")
for mode, key_v in (("base", "campaign"), ("reduce", "campaignReduce")):
    vv, pp = arr(key_v)
    for n in D["sweep"]:
        for is_two in (False, True):
            b = search(vv[:n], pp[:n], is_two)
            want = D["fits"][mode]["two" if is_two else "one"][str(n)]
            assert b["hp"] == want and b["margin"] > 1e-6, (mode, n, is_two)
print(f"every stored fit reproduced ({time.time() - t0:.0f} s)")

## 4. Scores on held-out records

- **RMSE**: the root mean square miss of the mean, in kW.
- **Coverage**: the share of held-out records inside the central 50, 80, 90 and 95% bands; chance alone moves a share near 90% by about ±3 points over 400 records (two standard errors).
- **Pinball loss**: for each quantile level $\tau = 0.05, \dots, 0.95$ of $N(m, \sigma^2)$, $\tau(y - q)$ above the quantile and $(1-\tau)(q - y)$ below, averaged. Lower is better; it rewards bands that are narrow and honest.

In [ ]:
TAU = np.arange(1, 20) / 20
ZT = stats.norm.ppf(TAU)

def scores(F, v, y):
    m, s = predict(F, v)
    e = y - m
    out = dict(rmse=np.sqrt(np.mean(e ** 2)), width=np.mean(2 * stats.norm.ppf(0.95) * s))
    out["cov"] = {l: np.mean(np.abs(e) <= stats.norm.ppf(0.5 + l / 200) * s) for l in (50, 80, 90, 95)}
    q = m[:, None] + s[:, None] * ZT[None, :]
    out["pinball"] = np.mean(np.where(y[:, None] >= q, TAU * (y[:, None] - q), (1 - TAU) * (q - y[:, None])))
    bands = [(3, 8), (8, 13), (13, 20.01)]
    out["by"] = [np.mean(np.abs(e[(v >= a) & (v < b)]) <= stats.norm.ppf(0.95) * s[(v >= a) & (v < b)]) for a, b in bands]
    return out

def check(s, r):
    assert abs(s["rmse"] - r["rmse"]) < 1e-6 and abs(s["pinball"] - r["pinball"]) < 1e-6 and abs(s["width"] - r["width"]) < 1e-6
    assert all(abs(s["cov"][l] - r["cov"][str(l)]) < 1e-12 for l in (50, 80, 90, 95)) and np.allclose(s["by"], r["cov90ByRange"], atol=1e-12)

s1, s2 = scores(one, v_test, p_test), scores(two, v_test, p_test)
check(s1, REF["scores"]["base"]["one"]); check(s2, REF["scores"]["base"]["two"])
for name, s in (("one source ", s1), ("two sources", s2)):
    print(f"{name}: RMSE {s['rmse']:.0f} kW, 90% band holds {100 * s['cov'][90]:.1f}% (by speed {', '.join(f'{100 * x:.0f}%' for x in s['by'])}), width {s['width']:.0f} kW, pinball {s['pinball']:.1f} kW")

**Read it.** One noise level for all wind speeds makes the band too wide at low wind and too narrow on the steep part (8–13 m/s), even when its overall coverage looks fine. Practice task 1 lets the noise change with wind speed.

## 5. More records, and a cheap curve that misleads

The cell scores both models for 16 to 64 records, then repeats the 16-record fits for a turbine with a load-reducing mode above 12.5 m/s that the maker's curve does not show.

In [ ]:
for n, r1, r2 in REF["sweepRMSE"]:
    a = fit(v_fit[:n], p_fit[:n], D["fits"]["base"]["one"][str(n)], False)
    b = fit(v_fit[:n], p_fit[:n], D["fits"]["base"]["two"][str(n)], True)
    sa, sb = scores(a, v_test, p_test)["rmse"], scores(b, v_test, p_test)["rmse"]
    assert abs(sa - r1) < 1e-6 and abs(sb - r2) < 1e-6
    print(f"{n:3d} records: RMSE one source {sa:5.0f} kW, two sources {sb:5.0f} kW, gap {sa - sb:4.0f} kW")

vr, pr = arr("campaignReduce"); vtr, ptr = arr("testReduce")
ra = scores(fit(vr[:N0], pr[:N0], D["fits"]["reduce"]["one"][str(N0)], False), vtr, ptr)
rb = scores(fit(vr[:N0], pr[:N0], D["fits"]["reduce"]["two"][str(N0)], True), vtr, ptr)
check(ra, REF["scores"]["reduce"]["one"]); check(rb, REF["scores"]["reduce"]["two"])
print(f"load-reducing mode, 13-20 m/s: one source holds {100 * ra['by'][2]:.0f}%, two sources {100 * rb['by'][2]:.0f}%; pinball {ra['pinball']:.1f} and {rb['pinball']:.1f} kW")

## 6. A standard library: scikit-learn

`GaussianProcessRegressor` fits the same kind of model with a continuous optimizer and restarts. It differs from the page in two ways: its mean is the records' average after `normalize_y` (not a fitted constant), and it searches settings continuously, not on a grid. Expect similar, not identical, numbers.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(3.0, (0.1, 50.0)) + WhiteKernel(0.01, (1e-6, 1.0))   # in units of the records' spread (normalize_y)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5, random_state=0)
gp.fit(v_fit[:N0, None], p_fit[:N0])
m, s = gp.predict(v_test[:, None], return_std=True)
rmse = np.sqrt(np.mean((p_test - m) ** 2)); cov90 = np.mean(np.abs(p_test - m) <= 1.645 * s)
print(gp.kernel_)
print(f"scikit-learn, 16 records: RMSE {rmse:.0f} kW, 90% band holds {100 * cov90:.1f}% (the page's one source: {s1['rmse']:.0f} kW, {100 * s1['cov'][90]:.1f}%)")
assert np.isfinite(rmse) and rmse < 400

## 7. Practice (about 60 minutes)

Write your own cells below each task. Predict first, then run, and report each score with the number of records it was fitted on and scored on.

1. **A noise level that changes with wind speed** (about 15 minutes). Give each record its own noise: for example 15 kW plus 0.3 m/s times the slope of the maker's curve (`np.gradient`), and pass it to `GaussianProcessRegressor(alpha=noise**2)` or add it to the diagonal of `K` in `fit`. Does the 90% band now hold about 90% in each of the three wind-speed ranges?
2. **Another kernel** (about 10 minutes). Replace the squared-exponential kernel with a Matérn kernel ($\nu = 5/2$, `sklearn.gaussian_process.kernels.Matern`). Which scores change, and why might a rougher kernel suit the knee at rated speed?
3. **Where to measure next** (about 10 minutes). Starting from 16 records, add one at a time the campaign record whose wind speed has the widest band (a simple form of active learning) instead of the next in time. After 8 additions, compare the held-out RMSE with the first 24 records in time order.
4. **A surrogate of a slow model** (about 15 minutes). Lesson M7's developer cost for a 24-month wait is $D = (mP + Ki)\,A$ with $i = (1+r)^{1/12} - 1$, $A = (1 - (1+i)^{-24})/i$, $P$ = 100 MW, $K$ = \$600M and $r$ = 8%: a function of the margin $m$ alone. Pretend each run takes an hour: run it at 5 margins between \$30,000 and \$60,000, fit a GP, then draw 1,000 uniform margins and compare the surrogate's P90 with the model's.
5. **Write it up** (about 10 minutes). In five sentences, report a two-source fit as a modeler would: the records and their range of wind speeds, the settings and how they were chosen, the held-out scores with their chance band, where the band fails, and what the maker's curve contributed.